# High-Pass Butterworth Filter for EEG

Applies a 1 Hz high-pass filter to the P4 channel of the local auditory EEG dataset.

**Dataset**: PhysioNet Auditory EEG (Abo Alzahab et al., 2021)  
**Channels**: P4, Cz, F8, T7  
**Sampling rate**: 1000 Hz

## 1. Install dependencies

In [ ]:
!pip install scipy numpy pandas matplotlib

## 2. Clone the resources repo (for data loader utility)

In [ ]:
import os
if not os.path.exists('python-EEG-Arabic-Resources'):
    !git clone https://github.com/NibrasAz7/python-EEG-Arabic-Resources.git
os.chdir('python-EEG-Arabic-Resources')

## 3. Download the local EEG dataset

In [ ]:
from pathlib import Path
data_dir = Path('data/local')
if not data_dir.exists() or not any(data_dir.glob('*.csv')):
    !python data/download_local.py

## 4. Apply high-pass filter

In [ ]:
from scipy import signal
import numpy as np
import matplotlib.pyplot as plt
from utils.eeg_loader import load_local_eeg

# Load local EEG data (subject 1, experiment 1, session 1)
timestamps, eeg_data, ch_names = load_local_eeg(
    data_dir='data/local', subject=1, experiment=1, session=1
)
channel_data = eeg_data[:, 0]  # P4 channel
fs = 1000  # Sampling rate (Hz)

# High-pass Butterworth filter at 1 Hz
def butter_highpass_filter(data, cutoff, fs, order=4):
    nyq = 0.5 * fs  # Nyquist frequency
    normal_cutoff = cutoff / nyq
    b, a = signal.butter(order, normal_cutoff, btype='high', analog=False)
    filtered = signal.filtfilt(b, a, data)
    return filtered

filtered_hp = butter_highpass_filter(channel_data, cutoff=1.0, fs=fs)

# Plot before and after
n_plot = 5000
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
axes[0].plot(timestamps[:n_plot], channel_data[:n_plot], label='Raw')
axes[0].set_ylabel('EEG (uV)')
axes[0].set_title('Before high-pass filter')
axes[1].plot(timestamps[:n_plot], filtered_hp[:n_plot], label='Filtered', color='green')
axes[1].set_ylabel('EEG (uV)')
axes[1].set_xlabel('Time (ms)')
axes[1].set_title('After high-pass filter (1 Hz)')
plt.tight_layout()
plt.savefig('highpass_result.png', dpi=150)
plt.show()
print(f'Channels: {ch_names}')
print(f'Signal length: {len(channel_data)} samples ({len(channel_data)/fs:.2f} s)')